In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import rootutils

In [3]:
rootutils.setup_root(
    os.getcwd(), indicator=".project-root", dotenv=True, pythonpath=True, cwd=False
)

PosixPath('/data/gena-lm-mk-2/old_gena_lm/downstream_tasks/caduceus')

In [4]:
name = "kuleshov-group/caduceus-ph_seqlen-131k_d_model-256_n_layer-16"

In [5]:
import transformers

/data/gena-lm-mk-2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
caduceus = transformers.AutoModel.from_pretrained(name, trust_remote_code=True)

In [7]:
caduceus

Caduceus(
  (backbone): CaduceusMixerModel(
    (embeddings): CaduceusEmbeddings(
      (word_embeddings): Embedding(16, 256)
    )
    (layers): ModuleList(
      (0-15): 16 x Block(
        (norm): RMSNorm()
        (mixer): BiMambaWrapper(
          (mamba_fwd): Mamba(
            (in_proj): Linear(in_features=256, out_features=1024, bias=False)
            (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
            (act): SiLU()
            (x_proj): Linear(in_features=512, out_features=48, bias=False)
            (dt_proj): Linear(in_features=16, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=256, bias=False)
          )
          (mamba_rev): Mamba(
            (in_proj): Linear(in_features=256, out_features=1024, bias=False)
            (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
            (act): SiLU()
            (x_proj): Linear(in_features=512, out_feature

In [8]:
caduceus.backbone

CaduceusMixerModel(
  (embeddings): CaduceusEmbeddings(
    (word_embeddings): Embedding(16, 256)
  )
  (layers): ModuleList(
    (0-15): 16 x Block(
      (norm): RMSNorm()
      (mixer): BiMambaWrapper(
        (mamba_fwd): Mamba(
          (in_proj): Linear(in_features=256, out_features=1024, bias=False)
          (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
          (act): SiLU()
          (x_proj): Linear(in_features=512, out_features=48, bias=False)
          (dt_proj): Linear(in_features=16, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=256, bias=False)
        )
        (mamba_rev): Mamba(
          (in_proj): Linear(in_features=256, out_features=1024, bias=False)
          (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
          (act): SiLU()
          (x_proj): Linear(in_features=512, out_features=48, bias=False)
          (dt_proj): Linear(in_features=16, ou

In [9]:
caduceus.config

CaduceusConfig {
  "architectures": [
    "CaduceusForMaskedLM"
  ],
  "auto_map": {
    "AutoConfig": "configuration_caduceus.CaduceusConfig",
    "AutoModel": "modeling_caduceus.Caduceus",
    "AutoModelForMaskedLM": "modeling_caduceus.CaduceusForMaskedLM",
    "AutoModelForSequenceClassification": "modeling_caduceus.CaduceusForSequenceClassification"
  },
  "bidirectional": true,
  "bidirectional_strategy": "add",
  "bidirectional_weight_tie": true,
  "complement_map": {
    "0": 0,
    "1": 1,
    "10": 7,
    "11": 11,
    "12": 12,
    "13": 13,
    "14": 14,
    "15": 15,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 10,
    "8": 9,
    "9": 8
  },
  "d_model": 256,
  "dtype": "float32",
  "fused_add_norm": true,
  "initializer_cfg": {
    "initializer_range": 0.02,
    "n_residuals_per_layer": 1,
    "rescale_prenorm_residual": true
  },
  "model_type": "caduceus",
  "n_layer": 16,
  "norm_epsilon": 1e-05,
  "pad_vocab_size_multiple": 8,
  "rcps": false,


In [10]:
import hydra

In [11]:
!export GENALM_HOME=/data/genalm

In [12]:
os.environ["GENALM_HOME"]="/data/genalm"

In [13]:
with hydra.initialize(version_base=None, config_path="configs/"):
    cfg = hydra.compose(config_name="v2_qnorm_large_mouse.yaml")

In [14]:
dataset = hydra.utils.instantiate(cfg.training_dataset)

In [15]:
sample = dataset[335]
print(sorted(sample.keys()))

['chrom', 'dataset_description', 'desc_vectors', 'end', 'gene_id', 'input_ids', 'labels', 'labels_mask', 'name', 'reverse', 'selected_keys', 'start']


In [16]:
sample["desc_vectors"].shape

torch.Size([14, 768])

In [17]:
sample["labels"].shape, sample["labels_mask"].shape, sample["input_ids"].shape, sample["desc_vectors"].shape

(torch.Size([504, 14]),
 torch.Size([504, 14]),
 torch.Size([504]),
 torch.Size([14, 768]))

In [18]:
from model.model import CaduceusExpressionCountsModel

In [19]:
from tqdm.autonotebook import tqdm

In [20]:
collator = hydra.utils.instantiate(cfg.collate_fn)

In [21]:
import torch

In [22]:
dataloader = torch.utils.data.DataLoader(
    dataset = dataset,
    num_workers = 1,
    shuffle = False,
    batch_size = 2,
    collate_fn = collator
)

In [23]:
for i, cpu_batch in enumerate(tqdm(dataloader)):
    cuda_batch = dict()
    for key, value in cpu_batch.items():
        if torch.is_tensor(value):
            cuda_batch[key] = value.to(device = "cuda")
        else:
            cuda_batch[key] = value
    if 16 < i:
        break

  0%|          | 0/7803 [00:00<?, ?it/s]

  0%|          | 17/7803 [00:00<03:01, 42.94it/s]


In [24]:
cfg_model = hydra.utils.instantiate(cfg.model).to(device = "cuda")

In [25]:
cfg_model

CaduceusExpressionCountsModel(
  (embeddings): CaduceusEmbeddings(
    (word_embeddings): Embedding(16, 256)
  )
  (norm_f): RMSNorm()
  (layers): ModuleList(
    (0-15): 16 x FusedLayer(
      (caduceus): Block(
        (norm): RMSNorm()
        (mixer): BiMambaWrapper(
          (mamba_fwd): Mamba(
            (in_proj): Linear(in_features=256, out_features=1024, bias=False)
            (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
            (act): SiLU()
            (x_proj): Linear(in_features=512, out_features=48, bias=False)
            (dt_proj): Linear(in_features=16, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=256, bias=False)
          )
          (mamba_rev): Mamba(
            (in_proj): Linear(in_features=256, out_features=1024, bias=False)
            (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
            (act): SiLU()
            (x_proj): Linear

In [26]:
torch.cuda.empty_cache()

In [27]:
cuda_batch["labels"].shape

torch.Size([2, 504, 14])

In [37]:
with torch.amp.autocast("cuda", dtype = torch.bfloat16):
    result = cfg_model(
        labels = cuda_batch["labels"],
        labels_mask = cuda_batch["labels_mask"],
        input_ids = cuda_batch["input_ids"],
        desc_vectors = cuda_batch["desc_vectors"],
    )

AcceleratorError: CUDA error: unspecified launch failure
Search for `cudaErrorLaunchFailure' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
result

ExpressionCountsModelOutput(loss=None, logits=tensor([[[ 0.4648],
         [-0.2490],
         [ 0.4648],
         ...,
         [ 0.2480],
         [ 0.3574],
         [-0.1011]],

        [[ 0.4590],
         [-0.2676],
         [ 0.4727],
         ...,
         [ 0.2637],
         [ 0.3574],
         [-0.0796]],

        [[ 0.4512],
         [-0.2539],
         [ 0.4590],
         ...,
         [ 0.2363],
         [ 0.3594],
         [-0.1001]],

        ...,

        [[ 0.1108],
         [ 0.4062],
         [-0.1543],
         ...,
         [-0.1211],
         [ 0.4199],
         [ 0.0947]],

        [[ 0.2754],
         [ 0.3418],
         [-0.2031],
         ...,
         [-0.1299],
         [ 0.4082],
         [ 0.1357]],

        [[ 0.2090],
         [ 0.4160],
         [-0.1367],
         ...,
         [-0.1133],
         [ 0.4199],
         [ 0.1230]]], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<ViewBackward0>), hidden_states=tensor([[[-1.6172e+00, -2.4121e-01, -1

In [30]:
result.logits.shape

torch.Size([28, 504, 1])

In [31]:
cuda_batch["labels"].shape

torch.Size([2, 504, 14])

In [34]:
cfg_model.losses(logits = result.logits, labels = cuda_batch["labels"], labels_mask = cuda_batch["labels_mask"])

Here


{'loss': tensor(11.4987, device='cuda:0', grad_fn=<DivBackward0>),
 'cls_loss': tensor(11.4987, device='cuda:0', grad_fn=<DivBackward0>),
 'other_loss': None,
 'labels_reshaped': tensor([[[0.0000e+00],
          [0.0000e+00],
          [0.0000e+00],
          ...,
          [0.0000e+00],
          [0.0000e+00],
          [4.7902e+00]],
 
         [[0.0000e+00],
          [0.0000e+00],
          [0.0000e+00],
          ...,
          [0.0000e+00],
          [0.0000e+00],
          [4.8394e+00]],
 
         [[0.0000e+00],
          [0.0000e+00],
          [0.0000e+00],
          ...,
          [0.0000e+00],
          [0.0000e+00],
          [4.7618e+00]],
 
         ...,
 
         [[0.0000e+00],
          [0.0000e+00],
          [0.0000e+00],
          ...,
          [0.0000e+00],
          [0.0000e+00],
          [1.3146e-02]],
 
         [[0.0000e+00],
          [0.0000e+00],
          [0.0000e+00],
          ...,
          [0.0000e+00],
          [0.0000e+00],
          [7.8978e-03]]